In [ ]:
# analysis 1

import os 
import glob
from tqdm import tqdm
import nibabel as nib
import numpy as np
import neuroimage_analysis as na
import matplotlib.pyplot as plt 
import pandas as pd
from scipy.stats import pearsonr


# Main Analysis Function

In [ ]:
dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

brain_template = nib.load(os.path.join(dir, "data/templates/Taylor_NHB_MNI152_T1_2mm_brain_mask_dil.nii.gz"))
brain_mask = brain_template.get_fdata() > 0


def get_slnm(fc_paths, behavior, visualize=True, outdir=None, filename=None):
    """
    Compute sLNM map by correlating FC maps with a behavior vector.
    If visualize=True, saves NIfTI files to outdir.
    """
    # Stack FC maps into (subjects x voxels)
    fc_array = np.hstack([nib.load(f).get_fdata()[brain_mask].reshape(-1, 1) for f in fc_paths]).T
    behavior = np.array(behavior).reshape(-1, 1)

    lnm_map = na.voxel_outcome_correlation(fc_array, behavior)

    if visualize:
        # Save NIfTI
        brain_3d = np.zeros_like(brain_template.get_fdata())
        brain_3d[brain_mask] = lnm_map
        lnm_img = nib.Nifti1Image(brain_3d, affine=brain_template.affine, header=brain_template.header)
        out_path = os.path.join(outdir, f'{filename}.nii.gz')
        nib.save(lnm_img, out_path)
        print(f'Saved: {out_path}')

    return lnm_map

def make_permutation_matrix(outcome_vector, n_perms):
    """Generate (subjects x n_perms) matrix with original + permuted outcomes."""
    n = len(outcome_vector)
    matrix = np.zeros((n, n_perms))
    matrix[:, 0] = outcome_vector
    for i in range(1, n_perms):
        matrix[:, i] = np.random.permutation(outcome_vector)
    return matrix


def permute_network(A_files, B_files, A_outcomes, B_outcomes, n_permutations=1000):
    """
    Generate permuted sLNM maps for datasets A and B.
    Returns permuted correlation maps of shape (voxels x n_permutations).
    """
    # Load FC arrays (subjects x voxels)
    A_fc_array = np.array([na.nifti_getdata(f) for f in A_files])
    B_fc_array = np.array([na.nifti_getdata(f) for f in B_files])

    # Generate permuted outcome matrices
    A_perm_outcomes = make_permutation_matrix(A_outcomes, n_perms=n_permutations)
    B_perm_outcomes = make_permutation_matrix(B_outcomes, n_perms=n_permutations)

    A_permuted_maps = na.voxel_outcome_correlation(A_fc_array, A_perm_outcomes)
    B_permuted_maps = na.voxel_outcome_correlation(B_fc_array, B_perm_outcomes)

    return A_permuted_maps, B_permuted_maps

# TMS Dataset

Data sourced from Weigand et al.[ *Biological Psychiatry* (2018)](https://www.biologicalpsychiatryjournal.com/article/S0006-3223(17)32158-3/abstract): "Prospective Validation That Subgenual Connectivity Predicts Antidepressant Efficacy of Transcranial Magnetic Stimulation Sites."

The study recorded MNI coordinates of TMS coil placement in depressive patients undergoing TMS treatment, along with post-treatment reduction in depressive scores (BDI).

Whole-brain FC maps were generated using tools from the [Taylor et al. (2023) repository](https://github.com/nimlab/NHB_Taylor2023): each subject's TMS target coordinate was converted into a spherical seed NIfTI using their [`spheremaker.ipynb`](https://github.com/nimlab/NHB_Taylor2023), and seed-based connectivity maps were computed using their [`connectomics.py`](https://github.com/nimlab/NHB_Taylor2023). All analyses use the MNI152 brain template provided in the same repository (also included here as `Taylor_NHB_MNI152_T1_2mm_brain_mask_dil.nii.gz`).

In [ ]:
# Load TMS dataset
tms_dataset = os.path.join(dir, "data/tms_dataset")

# Get FC files
tms_fc_files = sorted(glob.glob(os.path.join(tms_dataset, '*AvgR.nii.gz')))
print(f'Found {len(tms_fc_files)} TMS FC files')

# Load participant metadata
df_tms = pd.read_csv(os.path.join(tms_dataset, 'tms_patient_data.csv'))
df_tms = df_tms.sort_values('Patient').reset_index(drop=True)
print(f'Found {len(df_tms)} subjects')
print(df_tms[['Patient', 'BDI_change_percent']])

# additional analysis could be done using "HAMD_change_percent" instead which yields similar results

bdi_changed = (df_tms['BDI_change_percent']).tolist()

# Aphasia Recovery Cohort (ARC)

Data sourced from the open-access [Aphasia Recovery Cohort (ARC)](https://github.com/neurolabusc/AphasiaRecoveryCohortDemo) dataset (Gibson et al., [*Scientific Data*, 2024](https://www.nature.com/articles/s41597-024-03819-7)), which contains stroke lesion masks registered to standard MNI space and speech ability scores measured by the Western Aphasia Battery Aphasia Quotient (WAB-AQ).

We selected subjects with Broca's aphasia — the most common type in this dataset — who had an available lesion mask file (n = 85). Whole-brain FC maps were generated by using each subject's lesion mask as a seed and computing seed-based connectivity maps using [`connectomics.py`](https://github.com/nimlab/NHB_Taylor2023). All analyses use the MNI152 brain template included in the [Taylor et al. repository](https://github.com/nimlab/NHB_Taylor2023) (`Taylor_NHB_MNI152_T1_2mm_brain_mask_dil.nii.gz`).

In [ ]:
# Load and filter participant metadata
arc_dataset = os.path.join(dir, "data/arc_dataset")
df_arc = pd.read_csv(os.path.join(arc_dataset, 'participants.tsv'), sep='\t')
df_arc = df_arc.sort_values('participant_id').reset_index(drop=True)

# Get subjects with FC files
arc_fc_files = sorted(glob.glob(os.path.join(arc_dataset, '*sub*AvgR.nii.gz')))
arc_sub_fcid = [os.path.basename(f).split('_')[0].replace('w', '') for f in arc_fc_files]
print(f'Found {len(arc_fc_files)} subjects with FC files')

# Filter to Broca aphasia subjects with FC files
df_arc = df_arc[df_arc['participant_id'].isin(arc_sub_fcid)]
df_arc_broca = df_arc[df_arc['wab_type'] == 'Broca'].reset_index(drop=True)
print(f'Found {len(df_arc_broca)} Broca aphasia subjects')
print(df_arc_broca[['participant_id', 'wab_aq']])

# Get FC files for Broca subjects
sub_broca = df_arc_broca['participant_id'].tolist()
broca_fc_files = [f for f in arc_fc_files if os.path.basename(f).split('_')[0].replace('w', '') in sub_broca]
print(f'Found {len(broca_fc_files)} Broca FC files')

wab_aq = df_arc_broca['wab_aq'].tolist()

# sLNM analysis

In [ ]:
outdir = os.path.join(dir, "results")

tms_network = get_slnm(
    fc_paths=tms_fc_files,
    behavior=bdi_changed,
    visualize=True,
    outdir=outdir,
    filename='tms_network'
)

broca_network = get_slnm(
    fc_paths=broca_fc_files,
    behavior=wab_aq,
    visualize = True,
    outdir=outdir,
    filename='broca_network'
)

# Symptom-Permutation Testing

In [ ]:
empirical_r = pearsonr(broca_network, tms_network)[0]
print(f"Empirical similarity between sLNM maps: r = {empirical_r:.2f}")

n_permutations = 1000

# Permutation testing
broca_permuted_maps, tms_permuted_maps = permute_network(
    broca_fc_files,
    tms_fc_files,
    wab_aq,
    bdi_changed,
    n_permutations= n_permutations
)

permuted_r = []
sig_values = []
for i in range(n_permutations): 
    r = pearsonr(broca_permuted_maps[i, :], tms_permuted_maps[i, :])[0]
    permuted_r.append(r)
    if r >= empirical_r:
        sig_values.append(r)

p_value = len(sig_values) / (len(permuted_r) + 1)
print(f'Across {len(permuted_r)} permutations, {len(sig_values)} exceeded empirical r = {empirical_r:.2f}')
print(f'p = {p_value:.3f}')

# Plot
plt.rcParams['font.size'] = 18
plt.figure(figsize=(10, 8))
plt.hist(permuted_r, bins=40, alpha=0.7, color='steelblue', edgecolor='black')
plt.axvline(empirical_r, color='red', linestyle='--', linewidth=2.5, label=f'Empirical r = {empirical_r:.2f}')
plt.xlabel('Permuted Correlations')
plt.ylabel('Frequency')
plt.title('Symptom Permutation Test')
plt.legend()
plt.tight_layout()
plt.show()

# Partial Correlation Similarity Measures

In [ ]:
def partial_cor(A, B, C):
    # correlate A, B controlling for C
    r = np.corrcoef([A, B, C])
    r_AB, r_AC, r_BC = r[0,1], r[0,2], r[1,2]
    return (r_AB - r_AC * r_BC) / (np.sqrt(1 - r_AC**2) * np.sqrt(1 - r_BC**2))

pc1_map = na.nifti_getdata(os.path.join(dir, "data/pca/pca_voxelwise_pc1.nii.gz"))

# Empirical correlation controlling for PC1

r_emp_partial_pc1 = partial_cor(broca_network, tms_network, pc1_map)

# correlate each permuted map with PC1 (get rows of correlation values)
r_permuted_broca_pc1 = na.pearson_rows(broca_permuted_maps, pc1_map)
r_permuted_tms_pc1 = na.pearson_rows(tms_permuted_maps, pc1_map)

# correlate each pair of permuted maps 
r_permuted_broc_tms = na.pearson_rows(broca_permuted_maps, tms_permuted_maps)

# calculate all partial pc1 correlation at once
r_permuted_partial_pc1 = (r_permuted_broc_tms - r_permuted_broca_pc1 * r_permuted_tms_pc1) / ((np.sqrt(1 - r_permuted_broca_pc1**2))*(np.sqrt(1 - r_permuted_tms_pc1**2)))
# corrected p-value for PC1
p_pc1 = (np.sum(r_permuted_partial_pc1 >= r_emp_partial_pc1)+1)/(len(r_permuted_partial_pc1)+1)

print(f'Empirical partial r (PC1): {r_emp_partial_pc1:.3f}')
print(f'p-value (partial PC1):     {p_pc1:.3f}')

# Plot
plt.rcParams['font.size'] = 18
plt.figure(figsize=(10, 8))
plt.hist(r_permuted_partial_pc1, bins=40, alpha=0.7, color='steelblue', edgecolor='black')
plt.axvline(r_emp_partial_pc1, color='red', linestyle='--', linewidth=2.5, label=f'Empirical partial r_pc1 = {r_emp_partial_pc1:.2f}')
plt.xlabel('Permuted Correlations')
plt.ylabel('Frequency')
plt.title('Symptom Permutation Test')
plt.legend()
plt.tight_layout()
plt.show()
